In [2]:
import numpy as np
import pyspiel

game = pyspiel.load_game("tic_tac_toe")

In [6]:
class MCTSNode:
    def __init__(self, state: pyspiel.State = None, parent_action: int = None, parent=None):
        self.state: pyspiel.State = state
        self.parent_action = parent_action
        self.parent = parent
        self.children = {}
        self.visits = 0
        self.value = 0.0

        self.untried_actions = state.legal_actions() if state is not None else []
    
    def is_fully_expanded(self):
        return len(self.untried_actions) == 0

    def best_child(self, exploration_weight: float = 1.4):
        """
        UCB = (w_i / n_i) + c * sqrt(ln(N) / n_i)
        """
        best_score = float('-inf')
        best_child = None
        for action, child in self.children.items():
            exploit = child.value / child.visits
            explore = np.sqrt(np.log(self.visits) / child.visits)
            score = exploit + exploration_weight * explore
            if score > best_score:
                best_score = score
                best_child = child
        return best_child
    
class MCTS:
    def __init__(self, game: pyspiel.Game, exploration_weight: float = 1.4, iterations: int = 1000):
        self.game = game
        self.iterations = iterations
        self.exploration_weight = exploration_weight
    
    def search(self, initial_state: pyspiel.State):
        root = MCTSNode(state = initial_state)
        
        for _ in range(self.iterations):
            node = root
            # 1. selection
            while node.is_fully_expanded() and not node.state.is_terminal():
                node = node.best_child(self.exploration_weight)
            # 2. expansion
            if not node.state.is_terminal():
                action = node.untried_actions.pop()
                next_state = node.state.clone()
                next_state.apply_action(action)
                child_node = MCTSNode(state=next_state, parent_action=action, parent=node)
                node.children[action] = child_node
                node = child_node
            # 3. simulation (rollout)
            current_state = node.state.clone()
            while not current_state.is_terminal():
                legal_actions = current_state.legal_actions()
                action = np.random.choice(legal_actions)
                current_state.apply_action(action)
            # 4. backpropagation
            while node is not None:
                node.visits += 1
                if node.parent:
                    reward = current_state.returns()[node.parent.state.current_player()]
                    node.value += reward
                node = node.parent
            
        best_action = max(root.children.items(), key=lambda item: item[1].visits)[0]
        return best_action

In [8]:
mcts = MCTS(game, exploration_weight=1.4, iterations=1000)
wins, losses, draws = 0, 0, 0
num_games = 100
print(f'Running {num_games} games of MCTS vs Random Agent...')

for i in range(num_games):
    state = game.new_initial_state()
    while not state.is_terminal():
        if state.current_player() == 0:
            actions = mcts.search(state)
        else:
            actions = np.random.choice(state.legal_actions())
        state.apply_action(actions)
    result = state.returns()
    if result[0] > result[1]:
        wins += 1
    elif result[0] < result[1]:
        losses += 1
    else:
        draws += 1
    print(f'Game {i+1} finished. Result: {result}')
print(f'Out of {num_games} games: Wins: {wins}, Losses: {losses}, Draws: {draws}')

if losses == 0:
    print("MCTS Agent is unbeatable against Random Agent!")

Running 100 games of MCTS vs Random Agent...
Game 1 finished. Result: [1.0, -1.0]
Game 2 finished. Result: [1.0, -1.0]
Game 3 finished. Result: [1.0, -1.0]
Game 4 finished. Result: [1.0, -1.0]
Game 5 finished. Result: [1.0, -1.0]
Game 6 finished. Result: [1.0, -1.0]
Game 7 finished. Result: [1.0, -1.0]
Game 8 finished. Result: [1.0, -1.0]
Game 9 finished. Result: [1.0, -1.0]
Game 10 finished. Result: [1.0, -1.0]
Game 11 finished. Result: [1.0, -1.0]
Game 12 finished. Result: [1.0, -1.0]
Game 13 finished. Result: [1.0, -1.0]
Game 14 finished. Result: [1.0, -1.0]
Game 15 finished. Result: [1.0, -1.0]
Game 16 finished. Result: [1.0, -1.0]
Game 17 finished. Result: [1.0, -1.0]
Game 18 finished. Result: [1.0, -1.0]
Game 19 finished. Result: [1.0, -1.0]
Game 20 finished. Result: [1.0, -1.0]
Game 21 finished. Result: [1.0, -1.0]
Game 22 finished. Result: [1.0, -1.0]
Game 23 finished. Result: [1.0, -1.0]
Game 24 finished. Result: [1.0, -1.0]
Game 25 finished. Result: [1.0, -1.0]
Game 26 finish

In [10]:
# human vs AI
def human_vs_mcts():
    state = game.new_initial_state()
    mcts_agent = MCTS(game, exploration_weight=1.4, iterations=1000)
    print("Welcome to Tic Tac Toe! You are Player 1 (X).")
    while not state.is_terminal():
        if state.current_player() == 0:
            action = mcts_agent.search(state)
            print(f"MCTS Agent plays action: {action}")
        else:
            print(state)
            legal_actions = state.legal_actions()
            action = int(input(f"Your turn! Choose action from {legal_actions}: "))
            while action not in legal_actions:
                action = int(input(f"Invalid action. Choose action from {legal_actions}: "))
        state.apply_action(action)
    print(state)
    result = state.returns()
    if result[0] > result[1]:
        print("MCTS Agent wins!")
    elif result[0] < result[1]:
        print("You win!")
    else:
        print("It's a draw!")

human_vs_mcts()

Welcome to Tic Tac Toe! You are Player 1 (X).
MCTS Agent plays action: 4
...
.x.
...
MCTS Agent plays action: 1
ox.
.x.
...
MCTS Agent plays action: 3
ox.
xx.
.o.
MCTS Agent plays action: 6
ox.
xxo
xo.
MCTS Agent plays action: 8
oxo
xxo
xox
It's a draw!
